# PostgreSQL Catalog Tool Check

This notebook validates `ai-agent-tools/scripts/export_db_catalog.py`.

It will:
1. run the tool against local PostgreSQL,
2. load generated JSON,
3. show quick sanity checks for tables, views, and procedures/functions.

In [9]:
from pathlib import Path
import json
import subprocess

PROJECT_ROOT = Path.cwd().parent
SCRIPT_PATH = PROJECT_ROOT / "ai-agent-tools" / "scripts" / "export_db_catalog.py"
OUT_PATH = PROJECT_ROOT / "ai-agent-tools" / "configs" / "db_catalog_from_notebook.json"

print(f"Script: {SCRIPT_PATH}")
print(f"Output: {OUT_PATH}")

Script: /Users/nikolajabramov/PycharmProjects/Giga4DQM/ai-agent-tools/scripts/export_db_catalog.py
Output: /Users/nikolajabramov/PycharmProjects/Giga4DQM/ai-agent-tools/configs/db_catalog_from_notebook.json


In [10]:
# The script auto-loads PG* values from PROJECT_ROOT/.env by default.
cmd = [
    "uv", "run", "python", str(SCRIPT_PATH),
    "--schema", "s_grnplm_as_t_didsd_nnn_db_tmd",
    "--sections", "tables", "views", "procedures",
    "--pretty",
    "--output", str(OUT_PATH),
]

result = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
print("Exit code:", result.returncode)
if result.stdout.strip():
    print("STDOUT:\n", result.stdout)
if result.stderr.strip():
    print("STDERR:\n", result.stderr)

assert result.returncode == 0, "Catalog export failed"
assert OUT_PATH.exists(), "Output file was not created"
print("Catalog export completed.")

Exit code: 0
Catalog export completed.


In [11]:
catalog = json.loads(OUT_PATH.read_text(encoding="utf-8"))

tables = catalog.get("tables", [])
views = catalog.get("views", [])
procedures = catalog.get("procedures", [])

print("Schema:", catalog.get("schema"))
print("Tables:", len(tables))
print("Views:", len(views))
print("Procedures/Functions:", len(procedures))

if tables:
    print("\nSample table:", tables[0]["name"])
    print("Table description:", tables[0].get("description"))
    print("First columns:", [c["column_name"] for c in tables[0].get("columns", [])[:5]])

if views:
    print("\nSample view:", views[0]["name"])
    print("View description:", views[0].get("description"))

if procedures:
    print("\nSample routine:", procedures[0]["signature"])
    print("Routine kind:", procedures[0]["kind"])
    print("Routine description:", procedures[0].get("description"))

Schema: s_grnplm_as_t_didsd_nnn_db_tmd
Tables: 25
Views: 0
Procedures/Functions: 38

Sample table: d_agr_cred
Table description: Compact agreement-credit table used in local bootstrap mode.
First columns: ['agr_cred_id', 'prnt_agr_cred_id', 'info_system_id']

Sample routine: add_arr_log(in_workflow2_run_id bigint, strobject text, param text, sql_query text, INOUT vlog s_grnplm_as_t_didsd_nnn_db_tmd.tp_dm_log[])
Routine kind: function
Routine description: None


In [12]:
# Optional: inspect one DDL snippet
if tables:
    print(tables[0]["ddl"][:1200])

CREATE TABLE "s_grnplm_as_t_didsd_nnn_db_tmd"."d_agr_cred" (
    "agr_cred_id" uuid,
    "prnt_agr_cred_id" uuid,
    "info_system_id" smallint
);
